# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates step-by-step loading, inspection, and exploration of the FAIR<sup>2</sup> dataset ["Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells"](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/) and accessible at the following URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install -q mlcroissant

## 1. Data Loading

Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset (metadata only -- data is lazy-loaded by record set)
dataset = mlc.Dataset(croissant_url)

# Display brief dataset summary
meta = dataset.metadata
print("\033[1mDataset Title:\033[0m", getattr(meta, 'name', None))
print("\033[1mDescription:\033[0m", getattr(meta, 'description', None))
print("\033[1mDate Published:\033[0m", getattr(meta, 'datePublished', None))
print("\033[1mVersion:\033[0m", getattr(meta, 'version', None))
print("\033[1mIdentifier:\033[0m", getattr(meta, 'identifier', None))

## 2. Data Overview

List available record sets and their `@id`s, along with their respective fields and column IDs. All entity references use their Croissant `@id` for interoperability and traceability.

In [ ]:
record_sets = getattr(dataset.metadata, 'recordSet', None)
if not record_sets:
    print("No record sets found in this Croissant schema.")
else:
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', None)
        print(f"\nRecord Set: {rs_name} @id={rs_id}")
        fields = getattr(rs, 'field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            field_id = getattr(field, '@id', None)
            field_name = getattr(field, 'name', None)
            data_type = getattr(field, 'dataType', None)
            column_id = getattr(field, 'column', None)
            if isinstance(column_id, dict):
                column_id = column_id.get('@id')
            print(f"  Field: {field_name} @id={field_id} column={column_id} type={data_type}")


In [ ]:
# For demo purposes, let us collect record set @ids for extraction.
record_set_ids = []
if getattr(dataset.metadata, 'recordSet', None):
    for rs in dataset.metadata.recordSet:
        record_set_ids.append(getattr(rs, '@id', None))
else:
    print("No record sets available.")
pprint.pprint(record_set_ids)

## 3. Data Extraction

Extract records from each record set into a pandas DataFrame for further inspection and analysis. Use the primary record set's `@id` and field `@id`s as reference.

In [ ]:
dataframes = {}
# Select the main record set for analysis. If more than one, use the first for demo.
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"Loading records for main record set: {main_record_set_id}")
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df
    print(f"Columns available in record set {main_record_set_id}:")
    print(df.columns.tolist())
    display(df.head())
else:
    print("No record sets to extract data from.")

## 4. Exploratory Data Analysis (EDA)

Below are some common EDA operations: filtering, normalization, and grouping. We'll reference fields using their Croissant `@id`s. Please change the `numeric_field_id` and `group_field_id` to match actual fields discovered above.

In [ ]:
# ---- Edit these IDs as appropriate for your dataset ----
# Find a numeric field and a grouping field from the printed columns above
numeric_field_id = None # e.g., 'http://mlcommons.org/croissant/Abundance'
group_field_id = None   # e.g., 'http://mlcommons.org/croissant/Sample'

# Example: let's select the first float/integer column for demonstration
if len(dataframes) > 0:
    main_df = next(iter(dataframes.values()))
    # Try to pick numeric and group (categorical) fields automatically
    numeric_candidate = None
    group_candidate = None
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]) and numeric_candidate is None:
            numeric_candidate = col
        if pd.api.types.is_string_dtype(main_df[col]) and group_candidate is None:
            group_candidate = col
    if not numeric_field_id:
        numeric_field_id = numeric_candidate
    if not group_field_id:
        group_field_id = group_candidate

    print(f"Using numeric field (@id or column): {numeric_field_id}")
    print(f"Using group field (@id or column): {group_field_id}")

    # Filtering
    if numeric_field_id in main_df.columns:
        threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 0
        filtered_df = main_df[main_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouping
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print(f"Chosen numeric field '{numeric_field_id}' was not found in DataFrame columns.")
else:
    print("No records loaded; please verify record set and fields.")

## 5. Visualization

We plot the distribution of the selected numeric field and display an example group-wise summary. Update the field variables as necessary for your analytic goals.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Group bar plot
    if group_field_id in main_df.columns:
        plt.figure(figsize=(10,5))
        mean_by_group = main_df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
        mean_by_group.plot(kind='bar')
        plt.title(f"Mean of {numeric_field_id} grouped by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()


## 6. Conclusion

In this notebook, we loaded and previewed the FAIR<sup>2</sup> proteomics dataset using `mlcroissant` by referencing all record and field entities via their Croissant `@id` values. We explored available fields, performed basic filtering and normalization, and visualized core data distributions.

For further work, consult the Croissant schema for richer metadata fields, perform domain-specific filtering or transformation, and expand analyses using the full set of record sets and fields.